# Hedonic Pricing Regression
**Thesis: What Makes Vietnamese Groceries Expensive?**

This notebook runs OLS hedonic regressions to estimate implicit price premiums for product attributes (brand, import origin, health claims, pack size) in Vietnamese online grocery data.

## Pipeline
1. Load `output/product_features.csv` (built by `src/build_product_dataset.py` + `src/nlp_features.py`)
2. Filter to products with extractable size (regression requires unit-normalized DV)
3. Run pooled OLS + per-category OLS
4. Robustness: 3 date snapshot averages
5. Diagnostics: VIF, cluster-robust SE, R² decomposition

In [15]:
import sys
sys.path.insert(0, '..')  # so 'src' and 'config' are importable

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)

## 1. Load Data

In [16]:
df = pd.read_csv('../output/product_features.csv')
print(f"Loaded: {len(df)} products across {df['parent_category'].nunique()} categories")
df.head(3)

Loaded: 2270 products across 10 categories


,product_name,unit,subcategory,parent_category,avg_final_price,avg_marked_price,days_observed,promo_rate,avg_discount_depth,is_branded,is_house_brand,import_origin,is_import,has_health_claim,has_freshness_claim,name_length,unit_type,pack_count,size_ml,size_g,ln_pack_size,price_per_100ml,price_per_100g,ln_price_per_unit,marked_price_per_100ml,marked_price_per_100g,ln_marked_price_per_unit
0,Bơ lạt Anchor gói 227g,Miếng,Bơ Sữa - Phô Mai,Sữa Tươi,124385.7143,137500.0000,63,0.5714,0.1669,1,0,Úc,1,0,0,5,count,1,NaN,227.0000,5.4250,NaN,54795.4688,10.9114,NaN,60572.6872,11.0116
1,Bơ lạt Paysan Breton hộp 250g,Hộp,Bơ Sữa - Phô Mai,Sữa Tươi,174000.0000,NaN,44,0.0000,NaN,0,0,domestic,0,0,0,6,count,1,NaN,250.0000,5.5215,NaN,69600.0000,11.1505,NaN,NaN,NaN
2,Bơ lạt Presiden hộp 250g,Hộp,Bơ Sữa - Phô Mai,Sữa Tươi,181000.0000,NaN,49,0.0000,NaN,0,0,domestic,0,0,0,5,count,1,NaN,250.0000,5.5215,NaN,72400.0000,11.1900,NaN,NaN,NaN


In [17]:
# Overview of key features
feature_cols = [
    'is_branded', 'is_import', 'is_house_brand',
    'has_health_claim', 'has_freshness_claim',
    'ln_pack_size', 'pack_count', 'name_length',
    'ln_price_per_unit', 'ln_marked_price_per_unit',
]
df[feature_cols].describe().round(3)

,is_branded,is_import,is_house_brand,has_health_claim,has_freshness_claim,ln_pack_size,pack_count,name_length,ln_price_per_unit,ln_marked_price_per_unit
count,2270.0000,2270.0000,2270.0000,2270.0000,2270.0000,2052.0000,2270.0000,2270.0000,2052.0000,884.0000
mean,0.1960,0.0670,0.0090,0.0540,0.0670,5.1660,2.1520,7.7830,9.8210,9.6840
std,0.3970,0.2510,0.0960,0.2260,0.2500,1.0850,5.7110,2.5300,1.0360,1.0150
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.6930,1.0000,1.0000,6.1540,7.6130
25%,0.0000,0.0000,0.0000,0.0000,0.0000,4.5000,1.0000,6.0000,9.0780,8.8670
50%,0.0000,0.0000,0.0000,0.0000,0.0000,5.2360,1.0000,8.0000,9.7860,9.6320
75%,0.0000,0.0000,0.0000,0.0000,0.0000,5.8200,1.0000,9.0000,10.4680,10.3400
max,1.0000,1.0000,1.0000,1.0000,1.0000,8.5170,130.0000,18.0000,13.5610,13.5340


## 2. Descriptive Statistics (Table 1)
Mean prices, promotion rates, and feature rates by category.

In [18]:
table1 = df.groupby('parent_category').agg(
    n_products=('product_name', 'count'),
    avg_final_price=('avg_final_price', 'mean'),
    avg_marked_price=('avg_marked_price', 'mean'),
    promo_rate=('promo_rate', 'mean'),
    avg_discount_pct=('avg_discount_depth', 'mean'),
    pct_branded=('is_branded', 'mean'),
    pct_import=('is_import', 'mean'),
    pct_health=('has_health_claim', 'mean'),
).round(3)

table1['avg_discount_pct'] = (table1['avg_discount_pct'] * 100).round(1)
table1['promo_rate']       = (table1['promo_rate'] * 100).round(1)
table1['pct_branded']      = (table1['pct_branded'] * 100).round(1)
table1['pct_import']       = (table1['pct_import'] * 100).round(1)
table1['pct_health']       = (table1['pct_health'] * 100).round(1)

print("Table 1: Descriptive Statistics by Category")
table1

Table 1: Descriptive Statistics by Category


,n_products,avg_final_price,avg_marked_price,promo_rate,avg_discount_pct,pct_branded,pct_import,pct_health
parent_category,,,,,,,,
Bánh Kẹo,583,63370.4440,95980.7930,16.5000,19.4000,11.8000,3.1000,3.6000
Chăm Sóc Bé,58,141954.1670,140428.5710,28.7000,16.9000,31.0000,0.0000,19.0000
Gia Vị,300,67941.5230,88917.7150,35.6000,15.7000,17.7000,4.3000,3.7000
Mì - Thực Phẩm Ăn Liền,221,67264.9640,41352.9910,31.5000,14.5000,38.9000,8.1000,0.0000
Rau - Củ - Trái Cây,248,67858.1060,80909.5730,13.1000,17.8000,0.0000,14.9000,0.4000
Sữa Tươi,343,80471.3830,58824.5690,33.4000,12.5000,53.9000,3.5000,19.8000
Thực Phẩm Chế Biến,172,55178.8420,53837.7040,16.2000,16.5000,0.0000,16.9000,1.7000
Thực Phẩm Khô,182,65080.4470,58794.3820,26.9000,15.1000,14.8000,3.8000,2.2000
Thực Phẩm Đông Lạnh,137,90576.1510,89990.2440,10.2000,16.3000,0.0000,13.1000,0.0000


## 3. Regression Sample
Keep products where size was extractable (required for unit-normalized DV).

In [19]:
# Primary sample: products with extractable unit price
reg_df = df.dropna(subset=['ln_price_per_unit', 'ln_marked_price_per_unit', 'ln_pack_size']).copy()
reg_df = reg_df[np.isfinite(reg_df['ln_price_per_unit'])].copy()

# Exclude extreme outliers (Winsorize at 1%/99%)
lo, hi = reg_df['ln_price_per_unit'].quantile([0.01, 0.99])
reg_df = reg_df[(reg_df['ln_price_per_unit'] >= lo) & (reg_df['ln_price_per_unit'] <= hi)]

print(f"Regression sample: {len(reg_df)} products ({len(reg_df)/len(df)*100:.1f}% of total)")
print(f"Coverage by category:")
print(reg_df.groupby('parent_category').size().rename('n_in_sample'))

Regression sample: 867 products (38.2% of total)
Coverage by category:
parent_category
Bánh Kẹo                  152
Chăm Sóc Bé                23
Gia Vị                    179
Mì - Thực Phẩm Ăn Liền     96
Rau - Củ - Trái Cây        19
Sữa Tươi                  208
Thực Phẩm Chế Biến         68
Thực Phẩm Khô              81
Thực Phẩm Đông Lạnh        41
Name: n_in_sample, dtype: int64


## 4. Pooled OLS — Final Price DV (Table 2a)

```
ln(final_price/unit) = α + β1·is_branded + β2·is_import + β3·is_house_brand
                      + β4·ln_pack_size + β5·pack_count
                      + β6·has_health_claim + β7·has_freshness_claim
                      + β8·name_length + subcategory_FE + ε
```
SEs clustered by subcategory.

In [20]:
FORMULA = (
    'ln_price_per_unit ~ '
    'is_branded + is_import + is_house_brand + '
    'ln_pack_size + pack_count + '
    'has_health_claim + has_freshness_claim + '
    'name_length + C(subcategory)'
)

model_final = smf.ols(FORMULA, data=reg_df).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df['subcategory']},
)
print(model_final.summary2())

                              Results: Ordinary least squares
Model:                       OLS                       Adj. R-squared:             0.638    
Dependent Variable:          ln_price_per_unit         AIC:                        1534.4917
Date:                        2026-03-17 17:24          BIC:                        1744.1534
No. Observations:            867                       Log-Likelihood:             -723.25  
Df Model:                    43                        F-statistic:                82.60    
Df Residuals:                823                       Prob (F-statistic):         7.36e-20 
R-squared:                   0.656                     Scale:                      0.32713  
--------------------------------------------------------------------------------------------
                                             Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
--------------------------------------------------------------------------------------------
Intercep

## 5. Pooled OLS — Marked Price DV (Table 2b)
Compare β coefficients with final_price model to see if promotions erode premiums.

In [21]:
FORMULA_MARKED = FORMULA.replace('ln_price_per_unit', 'ln_marked_price_per_unit')

model_marked = smf.ols(FORMULA_MARKED, data=reg_df).fit(
    cov_type='cluster',
    cov_kwds={'groups': reg_df['subcategory']},
)
print(model_marked.summary2())

                              Results: Ordinary least squares
Model:                    OLS                            Adj. R-squared:           0.619    
Dependent Variable:       ln_marked_price_per_unit       AIC:                      1591.8065
Date:                     2026-03-17 17:24               BIC:                      1801.4682
No. Observations:         867                            Log-Likelihood:           -751.90  
Df Model:                 43                             F-statistic:              82.41    
Df Residuals:             823                            Prob (F-statistic):       7.64e-20 
R-squared:                0.637                          Scale:                    0.34948  
--------------------------------------------------------------------------------------------
                                             Coef.  Std.Err.    z     P>|z|   [0.025  0.975]
--------------------------------------------------------------------------------------------
Intercep

## 6. Side-by-Side Coefficient Comparison (Table 2)
Marked price vs. final price premiums — shows whether promotions close the gap.

In [22]:
KEY_VARS = [
    'is_branded', 'is_import', 'is_house_brand',
    'ln_pack_size', 'pack_count',
    'has_health_claim', 'has_freshness_claim', 'name_length',
]

def extract_coefs(model, label):
    coef = model.params[KEY_VARS]
    pval = model.pvalues[KEY_VARS]
    se   = model.bse[KEY_VARS]
    stars = pval.map(lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else '')))
    result = pd.DataFrame({
        f'β ({label})': coef.round(4).astype(str) + stars,
        f'SE ({label})': se.round(4),
    })
    return result

t2 = pd.concat([
    extract_coefs(model_marked, 'marked'),
    extract_coefs(model_final,  'final'),
], axis=1)

# Add R2 row
r2_row = pd.DataFrame({
    'β (marked)': [f"{model_marked.rsquared_adj:.3f}"],
    'SE (marked)': [''],
    'β (final)':  [f"{model_final.rsquared_adj:.3f}"],
    'SE (final)':  [''],
}, index=['Adj R²'])

table2 = pd.concat([t2, r2_row])
print("Table 2: Hedonic Regression — Marked Price vs. Final Price")
print(f"N = {len(reg_df)}")
table2

Table 2: Hedonic Regression — Marked Price vs. Final Price
N = 867


,β (marked),SE (marked),β (final),SE (final)
is_branded,-0.3084***,0.0729,-0.3487***,0.0869
is_import,0.0534,0.1280,0.0371,0.1226
is_house_brand,-0.1041***,0.0376,-0.2967***,0.0354
ln_pack_size,-0.3066***,0.0383,-0.3186***,0.0403
pack_count,-0.0348**,0.0153,-0.034**,0.0159
has_health_claim,-0.1177,0.1185,-0.1113,0.1214
has_freshness_claim,0.0352,0.0786,0.0242,0.0865
name_length,-0.0092,0.0146,-0.0109,0.0136
Adj R²,0.619,,0.638,


## 7. Per-Category Regressions (Table 3)
Brand premium varies by category — test the Dairy vs. Veg_Fruit gradient.

In [23]:
FOCUS_CATS = ['Sữa các loại', 'Rau-Củ-Trái cây', 'Thực phẩm khô', 'Thực Phẩm Chế Biến']

# Use all categories with at least 20 products in regression sample
cat_counts = reg_df.groupby('parent_category').size()
eligible_cats = cat_counts[cat_counts >= 20].index.tolist()
print(f"Categories with ≥20 products in sample: {eligible_cats}")

SIMPLE_FORMULA = (
    'ln_price_per_unit ~ '
    'is_branded + is_import + is_house_brand + '
    'ln_pack_size + pack_count + '
    'has_health_claim + has_freshness_claim + name_length'
)

cat_results = {}
for cat in eligible_cats:
    sub = reg_df[reg_df['parent_category'] == cat]
    if sub['subcategory'].nunique() > 1:
        formula = SIMPLE_FORMULA + ' + C(subcategory)'
    else:
        formula = SIMPLE_FORMULA
    try:
        m = smf.ols(formula, data=sub).fit()
        cat_results[cat] = m
        print(f"{cat}: n={len(sub)}, adj_R²={m.rsquared_adj:.3f}")
    except Exception as e:
        print(f"{cat}: failed — {e}")

Categories with ≥20 products in sample: ['Bánh Kẹo', 'Chăm Sóc Bé', 'Gia Vị', 'Mì - Thực Phẩm Ăn Liền', 'Sữa Tươi', 'Thực Phẩm Chế Biến', 'Thực Phẩm Khô', 'Thực Phẩm Đông Lạnh']
Bánh Kẹo: n=152, adj_R²=0.157
Chăm Sóc Bé: n=23, adj_R²=0.875
Gia Vị: n=179, adj_R²=0.467
Mì - Thực Phẩm Ăn Liền: n=96, adj_R²=0.298
Sữa Tươi: n=208, adj_R²=0.687
Thực Phẩm Chế Biến: n=68, adj_R²=0.288
Thực Phẩm Khô: n=81, adj_R²=0.881
Thực Phẩm Đông Lạnh: n=41, adj_R²=0.592


In [24]:
# Compile brand + import premiums across categories
rows = []
for cat, m in cat_results.items():
    row = {'category': cat, 'n': int(m.nobs), 'adj_R2': round(m.rsquared_adj, 3)}
    for var in ['is_branded', 'is_import', 'is_house_brand', 'has_health_claim']:
        if var in m.params:
            row[var]          = round(m.params[var], 4)
            row[f'{var}_pval'] = round(m.pvalues[var], 4)
    rows.append(row)

table3 = pd.DataFrame(rows).set_index('category')
print("Table 3: Cross-Category Premium Estimates (ln_price_per_unit DV)")
table3

Table 3: Cross-Category Premium Estimates (ln_price_per_unit DV)


,n,adj_R2,is_branded,is_branded_pval,is_import,is_import_pval,is_house_brand,is_house_brand_pval,has_health_claim,has_health_claim_pval
category,,,,,,,,,,
Bánh Kẹo,152,0.1570,-0.2949,0.0432,-0.0381,0.9010,-0.0000,0.0831,-0.7915,0.0993
Chăm Sóc Bé,23,0.8750,0.3396,0.0088,0.0000,0.0000,0.0000,0.0000,0.0997,0.5236
Gia Vị,179,0.4670,-0.6136,0.0000,-0.1509,0.5181,-0.0000,0.0002,0.2710,0.4310
Mì - Thực Phẩm Ăn Liền,96,0.2980,-0.4139,0.0007,0.0569,0.7323,-0.0000,0.2522,0.0000,NaN
Sữa Tươi,208,0.6870,-0.1935,0.0127,0.2562,0.1327,0.0000,0.0010,-0.1067,0.2521
Thực Phẩm Chế Biến,68,0.2880,-0.0000,0.0032,-0.4489,0.0356,0.0000,0.1860,-0.2954,0.4760
Thực Phẩm Khô,81,0.8810,-0.6368,0.0000,0.0227,0.9331,-0.0000,0.0000,0.2314,0.4264
Thực Phẩm Đông Lạnh,41,0.5920,0.0000,0.0034,0.1953,0.2734,-0.0000,0.0000,0.0000,NaN


## 8. Diagnostics

In [25]:
# VIF — multicollinearity check
X_vif = reg_df[[
    'is_branded', 'is_import', 'is_house_brand',
    'ln_pack_size', 'pack_count',
    'has_health_claim', 'has_freshness_claim', 'name_length',
]].dropna()

X_vif_const = sm.add_constant(X_vif)
vif_df = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif_const.values, i+1) for i in range(X_vif.shape[1])],
})
print("VIF (rule of thumb: VIF > 10 indicates multicollinearity)")
print(vif_df.to_string(index=False))

VIF (rule of thumb: VIF > 10 indicates multicollinearity)
            feature    VIF
         is_branded 1.1268
          is_import 1.0213
     is_house_brand 1.0336
       ln_pack_size 1.0851
         pack_count 1.2290
   has_health_claim 1.1099
has_freshness_claim 1.1155
        name_length 1.3681


In [26]:
# Partial F-tests: brand block and health claim block
RESTRICTED_NO_BRAND = (
    'ln_price_per_unit ~ '
    'is_import + ln_pack_size + pack_count + '
    'has_health_claim + has_freshness_claim + name_length + C(subcategory)'
)
RESTRICTED_NO_HEALTH = (
    'ln_price_per_unit ~ '
    'is_branded + is_import + is_house_brand + '
    'ln_pack_size + pack_count + name_length + C(subcategory)'
)

m_no_brand  = smf.ols(RESTRICTED_NO_BRAND,  data=reg_df).fit()
m_no_health = smf.ols(RESTRICTED_NO_HEALTH, data=reg_df).fit()
m_full      = smf.ols(FORMULA, data=reg_df).fit()  # no cluster for F-test

# F-statistic: (ΔR²/Δk) / ((1-R²_full)/(n-k_full))
def f_test(r_full, r_restricted, k_added, n, k_full):
    return ((r_full - r_restricted) / k_added) / ((1 - r_full) / (n - k_full - 1))

n = len(reg_df)
k = m_full.df_model

f_brand  = f_test(m_full.rsquared, m_no_brand.rsquared,  2, n, k)  # is_branded + is_house_brand
f_health = f_test(m_full.rsquared, m_no_health.rsquared, 2, n, k)  # has_health + has_fresh

print(f"Partial F-test — Brand block (is_branded + is_house_brand): F={f_brand:.2f}")
print(f"Partial F-test — Health claim block: F={f_health:.2f}")
print(f"(Critical value at α=0.05, df_num=2: ~3.0)")

Partial F-test — Brand block (is_branded + is_house_brand): F=23.94
Partial F-test — Health claim block: F=0.84
(Critical value at α=0.05, df_num=2: ~3.0)


## 9. Robustness: 3 Date Snapshots
Re-run pooled regression on Dec 2025, Jan 2026, and Feb 2026 monthly averages separately.
Check if key coefficients are stable (β₁ brand premium should vary < 20%).

In [ ]:
import sys, importlib
sys.path.insert(0, '..')

from pathlib import Path
from src.utils import PROJECT_ROOT, extract_date
from src.nlp_features import extract_features

CATEGORIES = [
    "Dairy", "Baby_product", "Veg_Fruit", "Confectionary",
    "Dry_Food", "Egg_and_soy", "Frozen", "Instant_food",
    "Processed_food", "Spice",
]
NEEDED_COLS = {'product_name', 'unit', 'final_price', 'marked_price'}
MONTH_RANGES = {
    'Dec-2025': ('2025-12-01', '2025-12-31'),
    'Jan-2026': ('2026-01-01', '2026-01-31'),
    'Feb-2026': ('2026-02-01', '2026-02-28'),
}

def load_snapshot(month_start, month_end):
    """Build product-level dataset for a specific date window."""
    all_frames = []
    for cat in CATEGORIES:
        folder = PROJECT_ROOT / 'data' / cat
        csv_folder = folder / 'CSV'
        lookup_file = folder / 'cat_lookup_table.csv'
        frames = []
        for f in sorted(csv_folder.glob('*.csv')):
            d = extract_date(f.name)
            if not (month_start <= d <= month_end):
                continue
            try:
                df_ = pd.read_csv(f, usecols=lambda c: c in NEEDED_COLS)
                frames.append(df_)
            except Exception:
                pass
        if not frames:
            continue
        daily = pd.concat(frames, ignore_index=True)
        daily['final_price'] = pd.to_numeric(daily['final_price'], errors='coerce')
        daily['marked_price'] = pd.to_numeric(daily['marked_price'], errors='coerce')
        daily = daily[daily['final_price'] > 0]
        if lookup_file.exists():
            lkp = pd.read_csv(lookup_file, usecols=['product_name', 'subcategory', 'parent_category'])
            lkp = lkp.drop_duplicates(subset=['product_name'], keep='last')
            daily = daily.merge(lkp, on='product_name', how='left')
        else:
            daily['subcategory'] = 'Gia Vị'
            daily['parent_category'] = 'Gia Vị'
        grp = daily.groupby(['product_name', 'unit', 'subcategory', 'parent_category'], dropna=False)
        agg = grp.agg(
            avg_final_price=('final_price', 'mean'),
            avg_marked_price=('marked_price', 'mean'),
        ).reset_index()
        all_frames.append(agg)
    if not all_frames:
        return pd.DataFrame()
    return pd.concat(all_frames, ignore_index=True)


snapshot_models = {}
for month, (start, end) in MONTH_RANGES.items():
    snap = load_snapshot(start, end)
    if snap.empty:
        print(f"{month}: no data")
        continue
    snap_feat = extract_features(snap)
    snap_reg = snap_feat.dropna(subset=['ln_price_per_unit', 'ln_pack_size', 'subcategory'])
    snap_reg = snap_reg[np.isfinite(snap_reg['ln_price_per_unit'])]
    if len(snap_reg) < 30:
        print(f"{month}: too few obs ({len(snap_reg)})")
        continue
    try:
        m = smf.ols(FORMULA, data=snap_reg).fit(
            cov_type='cluster',
            cov_kwds={'groups': snap_reg['subcategory']},
        )
        snapshot_models[month] = m
        print(f"{month}: n={len(snap_reg)}, adj_R²={m.rsquared_adj:.3f}, β_brand={m.params.get('is_branded', float('nan')):.4f}")
    except Exception as e:
        print(f"{month}: {e}")

# Stability check: β_brand across months
brand_betas = {m: mod.params.get('is_branded', np.nan) for m, mod in snapshot_models.items()}
print("\nBrand premium stability across months:")
print(pd.Series(brand_betas))

## 10. Export Tables for Thesis

In [28]:
out = PROJECT_ROOT / 'output'

table1.to_csv(out / 'thesis_table1_descriptive.csv', encoding='utf-8-sig')
table2.to_csv(out / 'thesis_table2_pooled_regression.csv', encoding='utf-8-sig')
table3.to_csv(out / 'thesis_table3_category_regression.csv', encoding='utf-8-sig')
vif_df.to_csv(out / 'thesis_table_vif.csv', index=False, encoding='utf-8-sig')

print("Saved 4 tables to output/")

Saved 4 tables to output/
